# D'QUILANE v0.2 — Versi Balita

Notebook super simpel buat belajar. Code pendek, komen jelas, dijamin jalan.

## Target skill balita:
- Output bukan random murni lagi
- Beberapa kata mulai nyambung
- Loss turun dari ~5 ke <3
- Bisa generate kalimat pendek yang masuk akal

In [ ]:
# ===== CELL 1: INSTALL & CEK GPU =====
import subprocess, sys
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', 'numpy', '-q'])
    import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    print('Pakai CPU (lambat). Pastikan Settings -> Accelerator -> GPU')
    DEVICE = 'cpu'
    print('Kalau belum: Run -> Change run type -> GPU T4')

In [ ]:
# ===== CELL 2: CONFIG BALITA =====
# Ini semua angka yang tentuin seberapa gede otaknya.
# Naik pelan-pelan. Jangan loncat ke dewasa langsung.

CONFIG = {
    'vocab_size': 256,     # jumlah kata/token yang dikenal
    'd_model':    128,     # ukuran vektor internal (lebih kecil = lebih cepat)
    'n_heads':    4,       # jumlah attention head
    'd_ff':       512,    # ukuran FFN intermediate (biasanya 4x d_model)
    'n_layers':   4,      # kedalaman (virtual, parameter sharing)
    'max_seq':    64,     # panjang context maksimal
    'epochs':     1000,
    'batch_size': 16,
    'lr':         3e-4,
}

# estimasi ukuran
d = CONFIG['d_model']
v = CONFIG['vocab_size']
f = CONFIG['d_ff']
n = CONFIG['max_seq']
uniq = v*d + n*d + 4*d*d + 3*d*f + 3*d
print(f'Estimasi unique params: {uniq:,}')
print(f'Estimasi file size:    {uniq*4/1024:.1f} KB (fp32, tanpa quantize)')
print(f'Virtual depth:         {CONFIG["n_layers"]} (effective = {uniq * CONFIG["n_layers"]:,})')

In [ ]:
# ===== CELL 3: DATASET KECIL =====
# Mulai dari teks kecil dulu. Anak balita belajar dari sedikit kata dulu.
# Kalau udah jalan, baru tambah teks lebih banyak di cell ini.

TEXT = '''Indonesia adalah negara kepulauan.
Ibu kota Indonesia adalah Jakarta.
Bahasa Indonesia adalah bahasa resmi.
Mata uang Indonesia adalah rupiah.
Indonesia kaya akan sumber daya alam.
Penduduk Indonesia sangat beragam.
Setiap suku memiliki budaya yang unik.
Gunung berapi banyak di Indonesia.
Hutan tropis Indonesia terbesar di dunia.
Laut Indonesia kaya akan ikan.
Iklim Indonesia adalah tropis.
Musim hujan dan kemarau silih berganti.
Pertanian adalah mata pencaharian utama.
Padi ditanam di sawah oleh petani.
Nasi adalah makanan pokok Indonesia.
Pendidikan kunci masa depan bangsa.
Teknologi berkembang pesat di era modern.
Komputer dan internet ubah cara komunikasi.
Kecerdasan buatan menjadi tren global.
Model bahasa besar sedang populer sekarang.
Transformer arsitektur dasar model modern.
Attention mekanisme memahami konteks kalimat.
Parameter sharing hemat ukuran model.
ALBERT contoh model dengan parameter sharing.
Belajar setiap hari bikin hidup bermakna.
Kerja keras kunci pencapaian tujuan.
Indonesia akan terus maju dan berkembang.
Anak muda harapan masa depan bangsa.
Mari bangun negeri dengan semangat pantang menyerah.
'''
print(f'Corpus: {len(TEXT)} chars, {len(TEXT.split())} words')

In [ ]:
# ===== CELL 4: TOKENIZER CHAR-LEVEL =====
# Paling simpel: 1 char = 1 token. Cocok buat balita level.
# Vocab = semua karakter unik di TEXT.

chars = sorted(set(TEXT))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
# tambah special tokens
stoi['<pad>'] = len(stoi)
itos[len(itos)] = '<pad>'
PAD_ID = stoi['<pad>']

VOCAB = len(stoi)
CONFIG['vocab_size'] = VOCAB
print(f'Vocab size: {VOCAB} karakter')
print(f'Contoh: "I" -> {stoi["I"]}, "n" -> {stoi["n"]}, "d" -> {stoi["d"]}')

def encode(s):
    return [stoi[c] for c in s if c in stoi]

def decode(ids):
    return ''.join(itos[i] for i in ids if i != PAD_ID)

# test
print('Test encode:', encode('Indonesia'))
print('Test decode:', decode(encode('Indonesia')))

In [ ]:
# ===== CELL 5: SIAPIN DATA TRAINING =====
# Split jadi window: input = 64 char, target = 64 char (shift 1).
# Model belajar: kalau ada 'Indonesia adal', tebak 'ah'.

import torch

data = torch.tensor(encode(TEXT), dtype=torch.long)
print(f'Total tokens: {len(data)}')

def get_batch():
    # ambil batch random
    idx = torch.randint(0, len(data) - CONFIG['max_seq'] - 1, (CONFIG['batch_size'],))
    x = torch.stack([data[i:i+CONFIG['max_seq']] for i in idx])
    y = torch.stack([data[i+1:i+1+CONFIG['max_seq']] for i in idx])
    return x.to(DEVICE), y.to(DEVICE)

x, y = get_batch()
print(f'Batch shape: x={x.shape}, y={y.shape}')

In [ ]:
# ===== CELL 6: MODEL D'QUILANE (BALITA) =====
# Semua dari nol. Komen lengkap biar gampang dipahami.

import torch.nn as nn
import torch.nn.functional as F
import math

class Attention(nn.Module):
    """Multi-head causal self-attention. Intinya: tiap token lihat token lain."""
    def __init__(self, d, h):
        super().__init__()
        assert d % h == 0, f'd_model ({d}) harus kelipatan n_heads ({h})'
        self.h = h
        self.dh = d // h  # dim per head
        self.qkv = nn.Linear(d, 3*d, bias=False)  # gabung Q,K,V jadi 1 matrix (hemat)
        self.out = nn.Linear(d, d, bias=False)
    
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.h, self.dh)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)  # (3, B, h, T, dh)
        # attention scores: (B, h, T, T)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)
        # mask: token t hanya boleh lihat token <= t (causal)
        mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v  # (B, h, T, dh)
        out = out.transpose(1, 2).reshape(B, T, C)  # gabung head
        return self.out(out)

class FFN(nn.Module):
    """Feed-Forward Network (SwiGLU style). Olah tiap token mandiri."""
    def __init__(self, d, d_ff):
        super().__init__()
        # 3 layer terpisah. JANGAN pakai chunk(2) - sumber bug sebelumnya!
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.val  = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)  # balik ke d_model
    
    def forward(self, x):
        # SwiGLU: down(silu(gate(x)) * val(x))
        return self.down(F.silu(self.gate(x)) * self.val(x))

class Block(nn.Module):
    """1 layer transformer: attention + FFN, dengan residual + RMSNorm."""
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.norm1 = nn.RMSNorm(d)  # PyTorch 2.4+ punya nn.RMSNorm
        self.attn  = Attention(d, h)
        self.norm2 = nn.RMSNorm(d)
        self.ffn   = FFN(d, d_ff)
    
    def forward(self, x):
        x = x + self.attn(self.norm1(x))  # residual 1
        x = x + self.ffn(self.norm2(x))   # residual 2
        return x

class DQuilane(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        # embedding: token + posisi
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['d_model'])
        self.pos_emb = nn.Embedding(cfg['max_seq'], cfg['d_model'])
        # 1 block (dipakai berulang = parameter sharing)
        self.block = Block(cfg['d_model'], cfg['n_heads'], cfg['d_ff'])
        self.norm_f = nn.RMSNorm(cfg['d_model'])
        # output head (tied ke embedding = hemat param)
        self.head = nn.Linear(cfg['d_model'], cfg['vocab_size'], bias=False)
        self.head.weight = self.tok_emb.weight  # tie!
        # init weight
        self.apply(self._init)
    
    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0, std=0.02)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        # PARAMETER SHARING: 1 block dipakai n_layers kali
        for _ in range(self.cfg['n_layers']):
            x = self.block(x)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg['vocab_size']),
                                    targets.view(-1))
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new=50, temp=0.8, top_k=10):
        self.eval()
        for _ in range(max_new):
            # crop context kalau lebih dari max_seq
            x = idx if idx.size(1) <= self.cfg['max_seq'] else idx[:, -self.cfg['max_seq']:]
            logits, _ = self(x)
            logits = logits[:, -1, :] / temp
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

# buat model
model = DQuilane(CONFIG).to(DEVICE)
n = sum(p.numel() for p in model.parameters())
print(f'D\'QUILANE lahir!')
print(f'Unique params: {n:,}  (~{n*4/1024:.1f} KB)')
print(f'Virtual depth: {CONFIG["n_layers"]}')
print(f'Effective (no share): {n * CONFIG["n_layers"]:,}')

# smoke test: forward pass
x, y = get_batch()
logits, loss = model(x, y)
print(f'\nSmoke test OK!')
print(f'  input:  {x.shape}')
print(f'  logits: {logits.shape}')
print(f'  loss:   {loss.item():.4f}  (random = {math.log(VOCAB):.4f})')

In [ ]:
# ===== CELL 7: TRAINING =====
# Loop training: ambil batch -> forward -> hitung loss -> backward -> update weight
# Tunggu sampai loss mendekati 1.5 (dari random ~5.5)

import time
opt = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['epochs'])

model.train()
t0 = time.time()
losses = []
for epoch in range(CONFIG['epochs']):
    xb, yb = get_batch()
    opt.zero_grad()
    _, loss = model(xb, yb)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()
    losses.append(loss.item())
    if epoch % 100 == 0 or epoch == CONFIG['epochs'] - 1:
        print(f'epoch {epoch+1:4d}/{CONFIG["epochs"]}  loss={loss.item():.4f}  t={time.time()-t0:.1f}s')

print(f'\nTraining done in {time.time()-t0:.1f}s')
print(f'Final loss: {losses[-1]:.4f}  (started at {losses[0]:.4f})')

In [ ]:
# ===== CELL 8: TEST GENERATE =====
# Kasih prompt, lihat output

prompts = ['Indonesia', 'Belajar', 'Padi', 'Teknologi']
for p in prompts:
    ids = torch.tensor([encode(p)], dtype=torch.long, device=DEVICE)
    out = model.generate(ids, max_new=80, temp=0.7, top_k=10)
    text = decode(out[0].tolist())
    print(f'\n[{p}] ->')
    print(text)

In [ ]:
# ===== CELL 9: SAVE MODEL =====
import os
os.makedirs('output', exist_ok=True)
torch.save({
    'state_dict': model.state_dict(),
    'config': CONFIG,
    'stoi': stoi,
    'itos': {int(k): v for k, v in itos.items()},
}, 'output/dquilane_balita.pt')

size = os.path.getsize('output/dquilane_balita.pt') / 1024
print(f'Model saved: output/dquilane_balita.pt ({size:.1f} KB)')
print(f'\nDownload: klik tab Output di kanan -> download file')

## 📈 Roadmap Tumbuh Kembang

D'QUILANE punya tahap pertumbuhan:

| Tahap | Skill | Config | File Size |
|-------|-------|--------|-----------|
| **Bayi** (v0.1 sandbox) | Output random | d=96, depth=12, char=46 | 750 KB |
| **Balita** (v0.2 ini) | Mulai hafal kata | d=128, depth=4, char=80 | ~250 KB |
| **Anak** (v0.3) | Kalimat pendek masuk akal | d=256, depth=12, BPE 512 | ~2 MB |
| **Remaja** (v0.4) | Topik bisa diatur | d=512, depth=24, BPE 2K, SFT | ~20 MB |
| **Dewasa** (v1.0) | Production ready | d=768+, depth=48, BPE 8K+ | ~100 MB+ |

### Cara naik tahap:
1. **Naikkan TEXT** di cell 3 — tambah teks Indonesia sebanyak mungkin
2. **Naikkan epochs** di cell 2 — dari 1000 → 5000
3. **Kalau loss < 2**, naikkan `d_model` dari 128 → 256
4. **Kalau loss < 1.5**, naikkan `n_layers` dari 4 → 12 (gratis, gak nambah size!)

### Catatan penting:
- Loss random = ln(vocab_size). Kalau vocab=80, random loss ≈ 4.4
- Loss target balita: < 2 (mulai masuk akal)
- Loss target anak: < 1.5 (kalimat koheren)
- Kalau loss stuck / naik-turun: turunkan `lr` ke 1e-4 atau tambah batch_size